### Messages
- Messages are the objects containing
  - role - identifies the message type
  - content - actual data/content of the msg
  - Metadata - Optional field such as response Information, token usage..

In [7]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")  # Ensure your GROQ API key is set in the environment variables

# GROQ Model Integration
model=init_chat_model("groq:qwen/qwen3-32b")
model 

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002B02010DE80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002B02010E900>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
model.invoke("what is AI") #human Input => output,AiMsg

### Text Prompts
TextPrompts are the strings - ideal for straight forward generation tasks where you don't need to retain the conversation history.
`model.invoke("What is LangChain")`
- Text Prompt is used when:
  - You have a single standalone request
  - You dont need the conversation history
  - need of the minimal code complexity

### Message Prompts
where we can pass the list of messages to the model,
This allows for more complex interactions and conversations with the model, as we can maintain context across multiple messages.
Message Types
- SystemMessage: Represents a message from the system, often used to provide instructions or context to the model.
    - where we can set the tone, style, or any specific instructions/guidelines for the model to follow during the conversation.
- HumanMessage: Represents user input and interactions with the model.
    - contains text, images,audio, video, files, toolCalls etc.. that the user provides as input to the model.
- AIMessage: Responses generated by the model based on the input it receives. can be textContent, toolCalls etc..
- ToolMessage: Represents a message/output from a tool or external source.
    - used to pass the results of a single tool execution back to the model.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a Poetry expert"),
    HumanMessage(content="Write a poem about on AI."),
]
response = model.invoke(messages)
print(response.content)

In [ ]:
system_msg = SystemMessage(content="You are a coding assistant")
messages = [
    system_msg,
    HumanMessage("how do i create a REST API"),
]
response = model.invoke(messages)
print(response.content)

In [ ]:
### Detailed info to the LLM through SystemMessage

from langchain.messages import SystemMessage, HumanMessage
system_msg = SystemMessage(content="You are a senior AI architect, You will be provided with a programming question, and you will answer it in detail, providing code examples and reasoning when necessary.")
messages = [
    system_msg,
    HumanMessage("How do I create a REST API in JavaScript?"),
]
response = model.invoke(messages)
print(response.content)

In [14]:
## Human Message
human_msg = HumanMessage(
    content="Hello, How ru doing..?",
    name = 'Aawni',
    id = "msg_123"
)
response = model.invoke([human_msg])
print(response.content)

<think>
Okay, the user greeted me with "Hello, How ru doing..?" and used some informal expressions like "How ru doing" and some typos like "ru" for "are you". I need to respond politely and warmly. First, I should acknowledge their greeting and show I'm listening. Then, ask them how they're doing in return. I should keep the tone friendly and approachable. Maybe add an emoji to make it more personable. Also, offer assistance in case they need anything. Need to make sure the response is concise but not too short. Avoid using complex sentences. Check for any typos in my reply. Alright, let me put that together.
</think>

Hi there! 😊 I'm doing well, thanks for asking! How are you feeling today? I'd love to hear what's on your mind or help out with anything you need! 🌟


In [ ]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage
ai_msg = AIMessage(content="Hello, How can I assist you today?", name="AI_Assistant", id="msg_456")
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is the capital of France?", name="User", id="msg_789"),
    ai_msg,
    HumanMessage(content="Great, What's 2+2?", name="User", id="msg_790")
]
response = model.invoke(messages)
print(response.usage_metadata)
print(response.content)

In [ ]:
from langchain.messages import AIMessage, ToolMessage

ai_message = AIMessage(
    content = [],
    tool_calls = [
        {
            "name":"get_weather",
            "args":{
                "location":"India"
            },
            "id":"call_123"
        }
    ]
)

weather_result = "Sunny, 30°C"
tool_message = ToolMessage(
    content = weather_result,
    tool_call_id = "call_123"   #must match the tool call id in the ai_message
)

messages = [
    HumanMessage(content="What's the weather in India?"),
    ai_message,
    tool_message
]
response = model.invoke(messages)

print(tool_message.content) #ToolMessage is again sent to Model as Input
print(response.content) #Model can use the tool result to generate final response

Sunny, 30°C
